Silver Layer - Data Cleansing and Standardization

Cleans, deduplicates, and standardizes data from bronze layer.

In [0]:
CREATE OR REPLACE TABLE retail_lakehouse.silver.customers
USING DELTA
AS
SELECT
    CustomerID,
    CustomerName,
    Email,
    City,
    Address,
    LastUpdated
FROM (
    SELECT
        CAST(CustomerID AS INT) AS CustomerID,
        INITCAP(TRIM(CustomerName)) AS CustomerName,
        LOWER(TRIM(Email)) AS Email,
        TRIM(City) AS City,
        TRIM(Address) AS Address,
        TO_DATE(LastUpdated, 'dd-MM-yyyy') AS LastUpdated,
        ROW_NUMBER() OVER (
            PARTITION BY CustomerID
            ORDER BY TO_DATE(LastUpdated, 'dd-MM-yyyy') DESC
        ) AS rn
    FROM retail_lakehouse.bronze.customers
    WHERE CustomerID IS NOT NULL
) deduped
WHERE rn = 1;

CREATE OR REPLACE TABLE retail_lakehouse.silver.products 
USING DELTA
AS
SELECT
    ProductID,
    ProductName,
    Category,
    UnitPrice
FROM (
    SELECT
        CAST(ProductID AS INT) AS ProductID,
        TRIM(ProductName) AS ProductName,
        TRIM(Category) AS Category,
        CAST(UnitPrice AS DOUBLE) AS UnitPrice,
        ROW_NUMBER() OVER (
            PARTITION BY ProductID
            ORDER BY ProductID
        ) AS rn
    FROM retail_lakehouse.bronze.products
    WHERE TRY_CAST(UnitPrice AS DOUBLE) > 0
) deduped
WHERE rn = 1;

CREATE OR REPLACE TABLE retail_lakehouse.silver.stores
USING DELTA
AS
SELECT
    StoreID,
    StoreName,
    Region
FROM (
    SELECT
        CAST(StoreID AS INT) AS StoreID,
        INITCAP(TRIM(StoreName)) AS StoreName,
        TRIM(Region) AS Region,
        ROW_NUMBER() OVER (
            PARTITION BY StoreID
            ORDER BY StoreID
        ) AS rn
    FROM retail_lakehouse.bronze.stores
    WHERE Region IS NOT NULL
) deduped
WHERE rn = 1;

CREATE OR REPLACE TABLE retail_lakehouse.silver.sales
USING DELTA
AS
SELECT
    TransactionID,
    CustomerID,
    ProductID,
    StoreID,
    Quantity,
    TxnDate
FROM (
    SELECT
        CAST(TransactionID AS INT) AS TransactionID,
        CAST(CustomerID AS INT) AS CustomerID,
        CAST(ProductID AS INT) AS ProductID,
        CAST(StoreID AS INT) AS StoreID,
        CAST(Quantity AS INT) AS Quantity,
        TO_DATE(TxnDate, 'dd-MM-yyyy') AS TxnDate,
        ROW_NUMBER() OVER (
            PARTITION BY TransactionID
            ORDER BY TO_DATE(TxnDate, 'dd-MM-yyyy') DESC
        ) AS rn
    FROM retail_lakehouse.bronze.sales
    WHERE TRY_CAST(Quantity AS INT) > 0
) deduped
WHERE rn = 1;

In [0]:
SELECT COUNT(*) FROM retail_lakehouse.silver.customers;
SELECT COUNT(*) FROM retail_lakehouse.silver.products;
SELECT COUNT(*) FROM retail_lakehouse.silver.stores;
SELECT COUNT(*) FROM retail_lakehouse.silver.sales;


**Enable CDC**

In [0]:
-- ALTER TABLE retail_lakehouse.silver.customers
-- SET TBLPROPERTIES (
--     delta.enableChangeDataFeed = true
-- );

-- ALTER TABLE retail_lakehouse.silver.sales
-- SET TBLPROPERTIES (
--     delta.enableChangeDataFeed = true
-- );


**Validating silver layer**

In [0]:
SELECT COUNT(*) AS customer_count
FROM retail_lakehouse.silver.customers;

SELECT COUNT(*) AS product_count
FROM retail_lakehouse.silver.products;

SELECT COUNT(*) AS store_count
FROM retail_lakehouse.silver.stores;

SELECT COUNT(*) AS sales_count
FROM retail_lakehouse.silver.sales;

SELECT *
FROM retail_lakehouse.silver.sales
WHERE Quantity <= 0;

SELECT *
FROM retail_lakehouse.silver.products
WHERE UnitPrice <= 0;


SELECT *
FROM retail_lakehouse.silver.sales
WHERE TransactionID IS NULL;

SELECT *
FROM retail_lakehouse.silver.customers
WHERE CustomerID IS NULL;

SELECT
    TransactionID,
    COUNT(*)
FROM retail_lakehouse.silver.sales
GROUP BY TransactionID
HAVING COUNT(*) > 1;

-- Proper Case Validation

SELECT CustomerName
FROM retail_lakehouse.silver.customers
WHERE CustomerName != INITCAP(CustomerName);

-- Lowercase Validation

SELECT Email
FROM retail_lakehouse.silver.customers
WHERE Email != LOWER(Email);

-- Date Validation

SELECT TxnDate
FROM retail_lakehouse.silver.sales
LIMIT 10;

**CDC Validation**

In [0]:
-- DESCRIBE HISTORY retail_lakehouse.silver.sales;
-- DESCRIBE HISTORY retail_lakehouse.silver.customers;